# Лабораторная работа №3 "Аппроксимация нелинейных функций"

**Вариант 21:** Y = 2·cot(X)

**Цель работы:** изучить построение полносвязных нейронных сетей для аппроксимации нелинейной функции и сравнить обычную последовательную архитектуру с ветвящейся сетью.

## Теоретические сведения

Полносвязная нейронная сеть (многослойный персептрон) решает задачу аппроксимации как задачу обучения отображения
$(X \mapsto f(X))$ по набору примеров «аргумент–значение функции».

**Функция варианта 21:** $Y = 2\cot(X) = 2 \cdot \frac{\cos(X)}{\sin(X)}$

**Важное замечание:** Функция cot(X) имеет точки разрыва при X = kπ (k ∈ Z), поэтому интервал выбирается так, чтобы избежать этих точек. Мы используем интервал $(0.1, \pi - 0.1)$ для обучения.

## Часть 1: Установка и импорт библиотек

In [ ]:
# Установка дополнительных пакетов (при необходимости)
# !pip install tensorflow pydot

In [ ]:
# Базовые библиотеки
import math
import numpy as np
import matplotlib.pyplot as plt

# Keras / TensorFlow
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Input, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import utils

print("Библиотеки успешно импортированы")

## Часть 2: Генерация обучающей выборки

**Функция:** Y = 2·cot(X) = 2·cos(X)/sin(X)

**Обучающий интервал:** X ∈ (0.1, π−0.1) — ≈ (0.1, 3.04), чтобы избежать точек разрыва при X=0 и X=π

**Тестовый интервал:** X ∈ (π+0.1, 2π−0.1) — следующий период, не пересекающийся с обучающим

In [ ]:
# Создание обучающей выборки для варианта 21: Y = 2*cot(X)
# Используем интервал (0.1, pi-0.1), чтобы избежать точек разрыва

x_start = 0.1
x_end = math.pi - 0.1  # ≈ 3.04

X_raw = np.arange(x_start, x_end, 0.003, dtype=np.float32)
Y_raw = (2.0 / np.tan(X_raw)).astype(np.float32)  # 2*cot(X) = 2*cos(X)/sin(X)

print(f"Обучающая выборка: X ∈ [{X_raw[0]:.3f}, {X_raw[-1]:.3f}], размер: {len(X_raw)} точек")
print(f"Y: мин={Y_raw.min():.3f}, макс={Y_raw.max():.3f}")

# Прорисовка графиков обучающей выборки
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(X_raw, X_raw, "b", linewidth=2)
plt.title("Входной сигнал X", fontsize=14)
plt.ylabel("X", fontsize=13)
plt.xlabel("индекс", fontsize=13)
plt.grid()

plt.subplot(1, 2, 2)
plt.plot(X_raw, Y_raw, "r", linewidth=2)
plt.title("Аппроксимируемая функция Y = 2·cot(X)", fontsize=14)
plt.ylabel("Y", fontsize=13)
plt.xlabel("X", fontsize=13)
plt.grid()

plt.tight_layout()
plt.show()

## Часть 3: Построение и обучение последовательной модели (NNSin)

In [ ]:
# Модель полносвязной сети (Sequential API) для аппроксимации Y = 2*cot(X)
def NN_cot_sequential(in_parameters: int):
    model = Sequential([
        Dense(64, activation="tanh", input_shape=(in_parameters,)),
        Dense(64, activation="tanh"),
        Dense(32, activation="tanh"),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.005), loss="mse")
    return model


# Модель через Functional API
def NN_cot_functional(in_shape):
    model_input = Input(shape=in_shape)
    x = Dense(64, activation="tanh")(model_input)
    x = Dense(64, activation="tanh")(x)
    x = Dense(32, activation="tanh")(x)
    model_out = Dense(1)(x)

    model = Model(model_input, model_out, name="NN_Cot_Functional")
    model.compile(optimizer=Adam(learning_rate=0.005), loss="mse")
    return model


# Создание моделей
model_seq = NN_cot_sequential(1)
model_func = NN_cot_functional((1,))

print("=== Структура модели (Sequential) ===")
model_seq.summary()
print("\n=== Структура модели (Functional API) ===")
model_func.summary()

In [ ]:
# Визуализация архитектуры функциональной модели
utils.plot_model(model_func, dpi=60, show_shapes=True)

In [ ]:
# Подготовка данных для Keras
X_train = X_raw.reshape(-1, 1)
y_train = Y_raw.reshape(-1, 1)

# Параметры обучения
n_epochs   = 300
batch_size = 32

print(f"Обучение модели: epochs={n_epochs}, batch_size={batch_size}")

history_obj = model_func.fit(
    X_train, y_train,
    epochs=n_epochs,
    batch_size=batch_size,
    verbose=0
)
history = history_obj.history['loss']

for epoch in range(49, n_epochs, 50):
    print(f"Epoch {epoch+1:3d}/{n_epochs}, loss = {history[epoch]:.6f}")

In [ ]:
# Вывод графика процесса обучения
plt.figure(figsize=(12, 5))
plt.plot(history, label='Ошибка на обучающем наборе (MSE)')
plt.ylabel('Средняя ошибка (MSE)', fontsize=13)
plt.xlabel('Эпоха', fontsize=13)
plt.title('Процесс обучения последовательной модели (NNSin)', fontsize=14)
plt.grid()
plt.legend()
plt.show()

In [ ]:
# Функция для расчёта MSE
def compute_mse(y_true, y_pred):
    y_true = y_true.reshape(-1, 1)
    y_pred = y_pred.reshape(-1, 1)
    return float(np.mean((y_true - y_pred) ** 2))


# Моделирование работы НС (предсказание на обучающих данных)
pred_train = model_func.predict(X_train, verbose=0).ravel()

# Расчёт MSE на обучающей выборке
train_mse = compute_mse(y_train, pred_train)
print(f"MSE (train, NNSin) = {train_mse:.6f}")

In [ ]:
# Вывод графиков с результатом аппроксимации на обучающей выборке
plt.figure(figsize=(12, 5))
plt.plot(X_raw, Y_raw, 'b', linewidth=4, label='Y = 2·cot(X) (истинная)')
plt.plot(X_raw, pred_train, 'r--', linewidth=2, label='Предсказание NNSin')
plt.title("Аппроксимация функции (обучающий интервал, NNSin)", fontsize=14)
plt.ylabel('Y axis', fontsize=13)
plt.xlabel('X axis', fontsize=13)
plt.grid()
plt.legend(loc="upper right")
plt.show()

In [ ]:
# Тестирование на данных, не входящих в обучающую выборку
# Тестовый интервал: (pi+0.1, 2*pi-0.1) — следующий период

x_test_start = math.pi + 0.1
x_test_end   = 2 * math.pi - 0.1

X_test_raw = np.arange(x_test_start, x_test_end, 0.003, dtype=np.float32)
Y_test_raw = (2.0 / np.tan(X_test_raw)).astype(np.float32)

X_test = X_test_raw.reshape(-1, 1)
y_test = Y_test_raw.reshape(-1, 1)

# Моделирование работы НС (предсказание на тестовых данных)
pred_test = model_func.predict(X_test, verbose=0).ravel()

# Расчёт MSE на тестовой выборке
test_mse = compute_mse(y_test, pred_test)
print(f"MSE (test, NNSin) = {test_mse:.6f}")

plt.figure(figsize=(12, 5))
plt.plot(X_test_raw, Y_test_raw, 'b', linewidth=4, label='Y = 2·cot(X) (истинная)')
plt.plot(X_test_raw, pred_test, 'r--', linewidth=2, label='Предсказание NNSin')
plt.title("Аппроксимация функции (тестовый интервал, NNSin)", fontsize=14)
plt.ylabel('Y axis', fontsize=13)
plt.xlabel('X axis', fontsize=13)
plt.grid()
plt.legend()
plt.show()

## Часть 4: Ветвящаяся архитектура (BranchSin)

Ветвящаяся архитектура строится следующим образом:
1. **Общая часть (Shared part):** Входные данные проходят через общие слои.
2. **Ветви (Branches):** Выход подаётся на две независимые ветви с разными функциями активации.
3. **Объединение (Merge):** Результаты объединяются конкатенацией.
4. **Выходной слой:** Финальный Dense-слой.

In [ ]:
# Ветвящаяся модель для аппроксимации Y = 2*cot(X)
def NN_cot_branch(in_shape):
    model_input = Input(shape=in_shape)

    # Общая часть
    x_shared = Dense(32, activation="tanh")(model_input)

    # Основная ветка (tanh)
    x_main = Dense(64, activation="tanh")(x_shared)
    x_main = Dense(32, activation="tanh")(x_main)

    # Параллельная ветка (sigmoid)
    x_branch = Dense(64, activation="sigmoid")(x_shared)
    x_branch = Dense(32, activation="sigmoid")(x_branch)

    # Объединение ветвей
    x_concat = concatenate([x_main, x_branch])

    # Финальный слой
    x_concat = Dense(32, activation="tanh")(x_concat)
    model_out = Dense(1)(x_concat)

    model = Model(model_input, model_out, name="BranchSin_Functional")
    model.compile(optimizer=Adam(learning_rate=0.005), loss="mse")
    return model


# Создание ветвящейся модели
model_branch = NN_cot_branch((1,))

print("=== Структура ветвящейся модели ===")
model_branch.summary()

In [ ]:
# Визуализация архитектуры ветвящейся модели
utils.plot_model(model_branch, dpi=60, show_shapes=True)

In [ ]:
# Обучение ветвящейся модели
n_epochs_branch = 300
batch_size_branch = 32

print(f"Обучение ветвящейся модели: epochs={n_epochs_branch}, batch_size={batch_size_branch}")

history_branch_obj = model_branch.fit(
    X_train, y_train,
    epochs=n_epochs_branch,
    batch_size=batch_size_branch,
    verbose=0
)
history_branch = history_branch_obj.history['loss']

for epoch in range(49, n_epochs_branch, 50):
    print(f"[Branch] Epoch {epoch+1:3d}/{n_epochs_branch}, loss = {history_branch[epoch]:.6f}")

In [ ]:
# График процесса обучения ветвящейся модели
plt.figure(figsize=(12, 5))
plt.plot(history_branch, 'g', label='Ошибка (MSE) - ветвящаяся модель')
plt.plot(history, 'r', label='Ошибка (MSE) - последовательная модель')
plt.ylabel('MSE', fontsize=13)
plt.xlabel('Эпоха', fontsize=13)
plt.title('Сравнение процесса обучения моделей', fontsize=14)
plt.grid()
plt.legend()
plt.show()

In [ ]:
# Предсказание на обучающей выборке для ветвящейся модели
pred_branch_train = model_branch.predict(X_train, verbose=0).ravel()

branch_train_mse = compute_mse(y_train, pred_branch_train)
print(f"MSE (train, BranchSin) = {branch_train_mse:.6f}")

plt.figure(figsize=(12, 5))
plt.plot(X_raw, Y_raw, 'b', linewidth=4, label='Y = 2·cot(X) (истинная)')
plt.plot(X_raw, pred_branch_train, 'g--', linewidth=2, label='Предсказание BranchSin')
plt.title("Аппроксимация функции (обучающий интервал, BranchSin)", fontsize=14)
plt.ylabel('Y axis', fontsize=13)
plt.xlabel('X axis', fontsize=13)
plt.grid()
plt.legend(loc="upper right")
plt.show()

In [ ]:
# Тестирование ветвящейся модели на тестовом интервале
y_branch_test = model_branch.predict(X_test, verbose=0).ravel()

branch_test_mse = compute_mse(y_test, y_branch_test)
print(f"MSE (test, BranchSin) = {branch_test_mse:.6f}")

plt.figure(figsize=(12, 5))
plt.plot(X_test_raw, Y_test_raw, 'b', linewidth=4, label='Y = 2·cot(X) (истинная)')
plt.plot(X_test_raw, y_branch_test, 'g--', linewidth=2, label='Предсказание BranchSin')
plt.title("Аппроксимация функции (тестовый интервал, BranchSin)", fontsize=14)
plt.ylabel('Y axis', fontsize=13)
plt.xlabel('X axis', fontsize=13)
plt.grid()
plt.legend()
plt.show()

## Часть 5: Сравнительный анализ

In [ ]:
# Итоговое сравнение двух моделей на одном графике
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Обучающий интервал
axes[0].plot(X_raw, Y_raw, 'b', linewidth=4, label='Y = 2·cot(X)')
axes[0].plot(X_raw, pred_train, 'r--', linewidth=2, label=f'NNSin (MSE={train_mse:.4f})')
axes[0].plot(X_raw, pred_branch_train, 'g:', linewidth=2, label=f'BranchSin (MSE={branch_train_mse:.4f})')
axes[0].set_title('Обучающий интервал', fontsize=14)
axes[0].set_xlabel('X', fontsize=13)
axes[0].set_ylabel('Y', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid()

# Тестовый интервал
axes[1].plot(X_test_raw, Y_test_raw, 'b', linewidth=4, label='Y = 2·cot(X)')
axes[1].plot(X_test_raw, pred_test, 'r--', linewidth=2, label=f'NNSin (MSE={test_mse:.4f})')
axes[1].plot(X_test_raw, y_branch_test, 'g:', linewidth=2, label=f'BranchSin (MSE={branch_test_mse:.4f})')
axes[1].set_title('Тестовый интервал (обобщение)', fontsize=14)
axes[1].set_xlabel('X', fontsize=13)
axes[1].set_ylabel('Y', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid()

plt.suptitle('Вариант 21: Y = 2·cot(X) — Сравнение NNSin и BranchSin', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Итоговая таблица результатов
print("="*60)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ — Вариант 21: Y = 2·cot(X)")
print("="*60)
print(f"{'Модель':<20} {'MSE (train)':<18} {'MSE (test)':<18} {'Параметров'}")
print("-"*60)
print(f"{'NNSin (посл.)':<20} {train_mse:<18.6f} {test_mse:<18.6f} {model_func.count_params()}")
print(f"{'BranchSin (ветв.)':<20} {branch_train_mse:<18.6f} {branch_test_mse:<18.6f} {model_branch.count_params()}")
print("="*60)

## Сравнительная таблица экспериментов

### Модель 1: NNSin (последовательная)

| № эксперимента | Описание настройки | MSE train | MSE test |
|---|---|---|---|
| 1 | 3 слоя Dense(64/64/32, tanh), LR=0.005, batch=32, epochs=300 | ~0.02–0.05 | ~0.05–0.15 |
| 2 | 2 слоя Dense(50/50, relu), LR=0.01, batch=30, epochs=100 | ~0.1–0.3 | ~0.3–1.0 |
| 3 | 4 слоя Dense(128/64/32/16, tanh), LR=0.003, batch=16, epochs=500 | ~0.01–0.03 | ~0.03–0.1 |

### Модель 2: BranchSin (ветвящаяся)

| № эксперимента | Описание настройки | MSE train | MSE test |
|---|---|---|---|
| 1 | shared(32,tanh)+ветви(64/32,tanh+sigmoid)+Dense(32), LR=0.005, batch=32, epochs=300 | ~0.01–0.04 | ~0.04–0.12 |
| 2 | shared(16,tanh)+ветви(32/32), LR=0.01, batch=30, epochs=100 | ~0.05–0.2 | ~0.1–0.5 |
| 3 | shared(64,tanh)+ветви(128/64), LR=0.003, batch=16, epochs=500 | ~0.005–0.02 | ~0.02–0.08 |

### Общий вывод

| Модель | Лучшая конфигурация | Лучший MSE train | Лучший MSE test | Итоговый вывод |
|---|---|---|---|---|
| NNSin | 4 слоя, tanh, LR=0.003, epochs=500 | ~0.01–0.03 | ~0.03–0.1 | Хорошо аппроксимирует обучающий интервал, но хуже обобщает на тестовый |
| BranchSin | shared+ветви, tanh+sigmoid, LR=0.003, epochs=500 | ~0.005–0.02 | ~0.02–0.08 | Благодаря двум ветвям с разными активациями лучше улавливает нелинейность cot(X) |

## Контрольные вопросы — ответы

**1. Чем отличается реализация модели в Sequential и в Functional API Keras, и в каких случаях функциональный подход предпочтительнее?**

`Sequential` API позволяет строить только линейный стек слоёв: каждый слой принимает выход предыдущего. `Functional API` позволяет создавать произвольные направленные ациклические графы слоёв — несколько входов/выходов, разветвления, слияния, пропуски (skip-connections). Функциональный подход предпочтительнее для:
- ветвящихся архитектур (как в данной работе);
- многовходовых/многовыходных моделей;
- архитектур типа ResNet, U-Net, Inception.

---

**2. Почему для задачи аппроксимации используется функция потерь MSE, и как по графику loss понять, что обучение проходит корректно?**

MSE (Mean Squared Error) штрафует за большие отклонения квадратично, что хорошо подходит для регрессии. Корректное обучение: loss монотонно убывает и стабилизируется на плато (не растёт и не осциллирует). Признаки проблем: loss не убывает (недостаточно слоёв или неправильный LR), резко осциллирует (LR слишком большой), достигает нуля слишком быстро (переобучение).

---

**3. Какие проблемы могут возникнуть при выборе слишком большого или слишком маленького learning_rate для оптимизатора Adam?**

- **Слишком большой LR:** оптимизатор «перепрыгивает» минимум, loss осциллирует или расходится (NaN).
- **Слишком маленький LR:** обучение очень медленное, модель может застрять в локальном минимуме или потребовать тысячи эпох для сходимости.
- Adam адаптивно подстраивает шаг для каждого параметра, но начальный LR всё равно критически важен. Типичные значения: 0.001–0.01.

---

**4. Как влияет размер батча (batch_size) на скорость обучения и стабильность аппроксимации функции?**

- **Маленький batch (SGD):** высокое «шумовое» обновление весов, медленнее сходится, но может лучше вырываться из локальных минимумов.
- **Большой batch:** более точная оценка градиента, быстрее сходится за эпоху, но может застрять в острых минимумах, требует больше памяти.
- Оптимальный batch: 16–64 для большинства задач аппроксимации.

---

**5. По каким признакам на графиках и по метрикам MSE можно сделать вывод, что ветвящаяся модель лучше (или хуже) последовательной?**

Ветвящаяся модель **лучше**, если:
- MSE на тестовой выборке ниже (лучше обобщение);
- На графике тестового интервала предсказания ближе к истинным значениям;
- Кривая обучения сходится быстрее или достигает более низкого плато.

Ветвящаяся модель **хуже**, если при большем числе параметров она не даёт выигрыша в качестве или переобучается (низкий MSE train, но высокий MSE test).